This notebook fits orthos to all replicates of the shendure-calibrated simulations

Imports

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

Autoreload for dev

In [ ]:
%load_ext autoreload
%autoreload 2

Create a nice large cluster. We will need the resorces.

In [ ]:
cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=4:00:00",
        f"--output=slave_%j.out"]
)
cluster.scale(jobs=5)
client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

Load the simulation object

In [ ]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
simu_obj_cell=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")

In [ ]:
#temporary : obj created w/ older version of code, necessitating this. 
simu_obj_cell.orthos=[]

Fit orthos to all replicates

In [ ]:
simu_obj_cell.create_orthos_for_all_replicates(client)

Save

In [ ]:
simu_obj_cell.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_with_orthos_20251008")

In [ ]:
simu_obj_cell.orthos[0].result()

Shut down the cluster

In [ ]:
client.close()
cluster.close()

# Move stuff below to its own nb...

Load the object twice, one for each set of hypothesies.

In [ ]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"
simu_obj_cell_type_hypotheses=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")
simu_obj_cre_hypotheses=scm.de_novo_simulation.load(client,path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251008")

Load the original ortho

In [ ]:
primordial=scm.ortho.load(client,f"{DATA_ROOT}/shendure","ortho_primordial_v3")

Let's start with the by cell type hypotheses

Create the hypothesis set

In [ ]:
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=primordial.training_data,
    reference_cre="reference",
    meta="emvar_screen",
)

Run wald on all reps

In [ ]:
simu_obj_cell_type_hypotheses._test_all_replicates(client,hs_all_ct,test="wald")